Splink is primarily a probabilistic record linkage library, meaning its main purpose is to find duplicate or link and merge related records in datasets that lack a common unique identifier

In [1]:
!pip install --upgrade pip

In [2]:
!pip install 'splink[spark]'

In [3]:
import splink, pyspark, duckdb, sqlite3
print("PySpark version:", pyspark.__version__)
print("Splink version:", splink.__version__)
print("DuckDB version:", duckdb.__version__)
print("SQLite3 version:", sqlite3.sqlite_version)

PySpark version: 3.5.5
Splink version: 4.0.6
DuckDB version: 1.2.0
SQLite3 version: 3.45.1


In [5]:
import json
import pandas as pd
import splink.comparison_library as cl
from splink import splink_datasets, SettingsCreator, block_on, Linker, DuckDBAPI
import datetime

# ========= Configure display settings for Jupyter Notebook ========= #
pd.options.display.max_columns = 1000
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

class DataLinkage:
    def __init__(self, df):
        """
        Initialize the DataLinkage class.
        :param df: The input DataFrame containing entity data for deduplication.
        """
        self.df = df.drop(columns=["cluster"], errors='ignore')  # Drop 'cluster' column if it exists
        self.settings = None
        self.linker = None

    # ========= Create settings for linkage ========= #
    def create_settings(self):
        self.settings = SettingsCreator(
            link_type="dedupe_only",  # Set deduplication mode
            comparisons=[
                cl.NameComparison("first_name"),  # Compare first names
                cl.NameComparison("surname"),  # Compare surnames
                cl.LevenshteinAtThresholds("dob", 1),  # Levenshtein distance on date of birth
                cl.ExactMatch("city").configure(term_frequency_adjustments=True),  # Exact match with frequency adjustments
                cl.EmailComparison("email"),  # Email comparison
            ],
            blocking_rules_to_generate_predictions=[
                block_on("first_name", "city"),  # Block on first name and city
                block_on("surname"),  # Block on surname
            ],
            retain_intermediate_calculation_columns=False,  # Set True to retain intermediate columns for debugging
        )
        self.linker = Linker(self.df, self.settings, db_api=DuckDBAPI())

    # ========= Train the model using blocking rules ========= #
    def train_model(self):
        deterministic_rules = [
            block_on("first_name", "dob"),  # Block on first name and DOB
            "l.first_name = r.first_name and levenshtein(r.surname, l.surname) <= 2",  # Fuzzy surname matching
            block_on("email"),  # Block on email for strong matches
        ]

        # Estimate probability that two random records match
        # Recall = 0.7 means we assume we can identify 70% of true matches
        # A higher recall increases sensitivity but may increase false positives
        self.linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)

        # Estimate u-probabilities using random sampling
        # max_pairs=1e6 ensures we sample up to 1 million record pairs to get a robust estimate, increase as necessary for huge dataset/s
        self.linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

        for rule in [block_on("first_name", "surname"), block_on("dob")]:
            # Estimate m-probabilities using Expectation Maximization (EM) algorithm
            # EM refines estimates iteratively for better match probability calculations
            self.linker.training.estimate_parameters_using_expectation_maximisation(rule)

    # ========= Save model settings with versioning ========= #
    def save_model(self):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")  # Generate timestamp for versioning
        file_path = f"../../results/saved_model_{timestamp}.json"  # Create versioned filename
        self.linker.misc.save_model_to_json(file_path, overwrite=True)
        print(f"Model saved as: {file_path}")
        return file_path

    # ========= Load an existing model ========= #
    def load_model(self, file_path):
        """
        Load a previously saved model.
        :param file_path: Path to the JSON model file.
        """
        with open(file_path, "r", encoding="utf-8") as f:
            self.settings = json.load(f)
        self.linker = Linker(self.df, self.settings, db_api=DuckDBAPI())

    # ========= Run inference and display results ========= #
    def predict(self, threshold_match_probability=0.2):
        """
        Predict matches and return a DataFrame with results.
        :param threshold_match_probability: Minimum probability to consider as a match.

        - A lower threshold (e.g., 0.2) increases sensitivity, capturing more possible matches.
        - A higher threshold (e.g., 0.9) reduces false positives but may miss true matches.
        """
        df_predictions = self.linker.inference.predict(threshold_match_probability=threshold_match_probability)
        clusters = self.linker.clustering.cluster_pairwise_predictions_at_threshold(
            df_predictions, threshold_match_probability=0.5  # Threshold for clustering, higher means stricter grouping
        )
        result_df = clusters.as_pandas_dataframe(limit=20)
        from IPython.display import display
        display(result_df)  # Display in Jupyter Notebook
        return result_df

if __name__ == "__main__":
    df = splink_datasets.fake_1000  # Load sample dataset
    linkage = DataLinkage(df)
    linkage.create_settings()
    linkage.train_model()
    model_path = linkage.save_model()  # Save with versioning
    linkage.load_model(model_path)
    result_df = linkage.predict()

Probability two random records match is estimated to be  0.00298.
This means that amongst all possible pairwise record comparisons, one in 335.56 are expected to match.  With 499,500 total possible comparisons, we expect a total of around 1,488.57 matching pairs
You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."first_name

Model saved as: ../../results/saved_model_20250318_192745.json


Predict time: 0.14 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'email':
    m values not fully trained
Completed iteration 1, num representatives needing updating: 2
Completed iteration 2, num representatives needing updating: 0


,cluster_id,unique_id,first_name,surname,dob,city,email
0,8,9,Evie,Dean,2015-03-03,Pootsmruth,evihd56@earris-bailey.net
1,14,14,Oliver,Griffiths,1991-10-26,Lunton,o.griffiths90@reyes-coleman.com
2,22,24,Thoas,Green,1974-10-05,London,thomas.green@clark.org
3,26,26,Thomas,Gabriel,1976-09-15,Loodon,gabriel.t54@nnichls.info
4,26,30,Thomas,Gabriel,1976-09-15,London,gabriel.t54@nlchois.info
5,37,37,Theodore,Morris,1978-08-19,Birmingham,t.m39@brooks-sawyer.com
6,37,39,Theodore,Morris,1978-08-19,Birmingham,t.m39@brooks-sawyer.com
7,37,43,Theodore,Morris,1978-08-19,Birmingham,t.m39@brooks-sawyer.com
8,52,52,Jyayden,Bnennet,2017-01-11,Snawseaa,jb88@king.com
9,74,74,Ronni,Begum,2003-10-15,London,r.b80@ellis-berry.com
